In [149]:
def chromaticities(lattice, mode, X):
#pack = [lattice, gamma, EB1, particle, sextupoles]
    chrom_code = f"""
    INCLUDE '{X[0]}';
    
    PROCEDURE MAIN;
      VARIABLE WHERE 100; VARIABLE MRKR 100;
      VARIABLE GAMMA 1; VARIABLE MAPSE 1;
      VARIABLE MU 800; VARIABLE NBAR 800 3; VARIABLE EL 1;
      VARIABLE MU0 1;
      VARIABLE PNUM 1;
      VARIABLE PSI0_DEG 1;
      VARIABLE NTURN 1;
      VARIABLE TUNE 800 3;
      VARIABLE MU_TP 800 3;
      VARIABLE CHROM_X 1; VARIABLE CHROM_Y 1;
      VARIABLE SGx1 1; VARIABLE SGy1 1; VARIABLE EB1 1;
      VARIABLE SGx2 1; VARIABLE SGy2 1;
      VARIABLE A 100 2 ; VARIABLE B 100 2 ; VARIABLE G 100 2 ;
      VARIABLE R 100 2 ; VARIABLE F 100 6 ;
      VARIABLE quadKx 1; VARIABLE quadKy 1; VARIABLE quadKz 1;
      VARIABLE OBJ 1;
    
      GROUTF 'img/dump/TR' 1;
    
    GAMMA:={X[1]};
    EB1 := {X[2]};
    
      SGx1 :=  {X[4][0]};
      SGy1  := {X[4][1]};
      SGx2 :=  {X[4][2]};
      SGy2  := {X[4][3]};
"""
    match (mode):
        case ("chromaticities"):
            chrom_code1 = f"""
                  OV 4 2 1;
                SET_FOR_{X[3]}_CHROM GAMMA;
              MRKR := 'chromaticities_{lattice}_{mode}';
              LATTICE SGx1 SGy1 SGx2 SGy2 EB1 1; PM 6;
                TP MU_TP ;
               WRITE 6 'DELTA DEPENDENT TUNES' MU_TP(1) MU_TP(2) ;
                CHROM_X:= (MU_TP(1)|(0&0&0&0&1))*(1+1/GAMMA);
                CHROM_Y:= (MU_TP(2)|(0&0&0&0&1))*(1+1/GAMMA);

            OV 3 3 0;
            SET_FOR_{X[3]} GAMMA;
            
            UM; LATTICE SGx1 SGy1 SGx2 SGy2 EB1 1;
            TSS MU NBAR 0;
            MU0 := CONS(MU);
            DAPEE MU 11 quadKx;
            DAPEE MU 33 quadKy;
            DAPEE MU 66 quadKz;
            OBJ := SQRT(SQR(quadKx) + SQR(quadKy)+SQR(quadKz));

            """
        case ("chrom_cor"):
            chrom_code1 = f"""
                  OV 4 2 1;
            SET_FOR_{X[3]}_CHROM GAMMA;
    
            FIT SGx1 SGy1 SGx2;
    
            LATTICE SGx1 SGy1 SGx2 SGy2 EB1 1;
            TP MU_TP;
            CHROM_X:= (MU_TP(1)|(0&0&0&0&1))*(1+1/GAMMA);
            CHROM_Y:= (MU_TP(2)|(0&0&0&0&1))*(1+1/GAMMA);
            OBJ:= SQRT(SQR(CHROM_X) + SQR(CHROM_Y));
    
            ENDFIT 1E-6 800 1 obj;
            MRKR := 'chrom_cor_{lattice}_{mode}';

            OV 3 3 0;
            SET_FOR_{X[3]} GAMMA;
            
            UM; LATTICE SGx1 SGy1 SGx2 SGy2 EB1 1;
            TSS MU NBAR 0;
            MU0 := CONS(MU);
            DAPEE MU 11 quadKx;
            DAPEE MU 33 quadKy;
            DAPEE MU 66 quadKz;
            OBJ := SQRT(SQR(quadKx) + SQR(quadKy)+SQR(quadKz));
            """
        case ("spin_coherence"):
            chrom_code1 = f"""
            OV 3 3 0;
            SET_FOR_{X[3]} GAMMA;
            
            FIT SGx1 SGy1 SGx2 ;
            LATTICE SGx1 SGy1 SGx2 SGy2 EB1 1;
            TSS MU NBAR 0;
            MU0 := CONS(MU);
            DAPEE MU 11 quadKx;
            DAPEE MU 33 quadKy;
            DAPEE MU 66 quadKz;
            OBJ := SQRT(SQR(quadKx) + SQR(quadKy)+SQR(quadKz));
            ENDFIT 1E-9 1000 1 OBJ;

            OV 3 2 1 ; 
                SET_FOR_{X[3]}_CHROM GAMMA ; 
                UM ; LATTICE SGx1 SGy1 SGx2 SGy2 EB1 1 ; 
                TP MU_TP ;
                
                CHROM_X := (MU_TP(1)|(0&0&0&0&1)) * (1+1/GAMMA) ;
                CHROM_Y := (MU_TP(2)|(0&0&0&0&1)) * (1+1/GAMMA) ;
            """
            
    ending = f"""
                OPENF 3618 'C:/Users/palo4/Desktop/MEPHI/Master/HNP/MIPT conf/COSY/tracking data/py.dat' 'REPLACE';
        
                   WRITE 3618 '#SGx1 		  SGx2 		  SGy1 		  chrom_x 		  chrom_y 		  quad_kx 		  quad_ky 		  quad_kz 		  obj';
                	WRITE 3618  ST(SGx1)&' '&ST(SGx2)&' '&ST(SGy1)&' '&ST(chrom_x)&' '&ST(chrom_y)&' '&ST(quadkx)&' '&ST(quadky)&' '&ST(quadkz)&' '&ST(obj);
                	CLOSEF 3618;
    
    ENDPROCEDURE;
    
    PROCEDURE RUN;
      MAIN;
    ENDPROCEDURE;
    RUN; END;
"""

    with open(f"py_{lattice}.fox", "w") as f:
        f.write(chrom_code)
        f.write(chrom_code1)
        f.write(ending)
    print(f"Расчет хроматичности для кольца {lattice} в режиме {mode}: {lattice}_{mode}", flush=True)

In [150]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def set_variables(lattice, mode):
    match (lattice):
        case ("magnetic_2p"):
            elenumber = 100
            lattice = "magnetic_2p"
            gamma = 1.143914
            EB1 = 0
            particle = "deuterons"
            sextupoles = [0,0,0,0]
            pack = [lattice, gamma, EB1, particle, sextupoles, elenumber]
            return(pack)

        case ("electrostatic"):
            elenumber = 540
            lattice = "electrostatic"
            gamma = 1.24810736
            EB1 = 112.464392
            particle = "protons"
            sextupoles = [0,0,0,0]
            pack = [lattice, gamma, EB1, particle, sextupoles, elenumber]
            return(pack)

def get_chromaticities(lattice, mode):
    variables = set_variables(lattice, mode)
    chromaticities(lattice, mode, variables)
    c = f"py_{lattice}.fox"
    return(c)
# 2nd argument for c:
#chromaticities: natural chromaticity
#chrom_cor: betatron chromaticity = 0
#spin_coherence: quads --> 0
c = get_chromaticities("magnetic_2p", "chromaticities")


Расчет хроматичности для кольца magnetic_2p в режиме chromaticities: magnetic_2p_chromaticities


In [153]:
import subprocess

subprocess.Popen(f'start cmd /c cosy {c}', shell=True)

<Popen: returncode: None args: 'start cmd /c cosy py_magnetic_2p.fox'>

In [154]:
with open("C:/Users/palo4/Desktop/MEPHI/Master/HNP/MIPT conf/COSY/tracking data/py.dat", "r") as f:
    lines = f.readlines()
print(lines[1])   
line = lines[1]
line1 = "#SGx1 		  SGx2 		  SGy1 		  chrom_x 		  chrom_y 		  quad_kx 		  quad_ky 		  quad_kz 		  obj"
with open(f"chrom_quad_{c}.dat", "a+") as f:
    f.seek(0)
    existing = f.readlines()
    if line1 + "\n" not in existing:
        f.write(line1 + "\n")

    if line not in existing:
        f.write(line if line.endswith("\n") else line + "\n")

 0.000000000000000  0.000000000000000  0.000000000000000 -1.879527862427126 -2.637586860764062 0.1029866600906237E-002 -.1947007313100617E-003 0.1236263991205405E-001 0.1240698993251572E-001

